In [135]:
import re
import json
from pathlib import Path
from collections import Counter
from typing import List, Dict, Any, Optional, Tuple

from openpyxl import load_workbook
from difflib import SequenceMatcher

import pymupdf

In [136]:
# =========================
# НАСТРОЙКИ
# =========================

PDF_PATH = Path(
    r"C:\Users\Aleksey\Desktop\Рабочие задачи\Задача к 19.06.2026\тест_структурного_парсера_v2\КР359_3.pdf"
)

REGISTRY_XLSX_PATH = Path(
    r"C:\Users\Aleksey\Desktop\Рабочие задачи\Задача к 19.06.2026\тест_структурного_парсера_v2\Список утвержденных клинических рекомендаций.xlsx"
)

OUTPUT_JSON_PATH = PDF_PATH.with_suffix(".json")

In [137]:
# =========================
# ОБЩИЕ УТИЛИТЫ
# =========================

MAIN_HEADING_PREFIXES = (
    "краткая информация",
    "диагностика",
    "лечение",
    "медицинская реабилитация",
    "реабилитация",
    "профилактика",
    "организация оказания медицинской помощи",
    "дополнительная информация",
    "критерии оценки качества",
)

FORBIDDEN_HEADING_STARTS = (
    "+",
    "-",
    "•",
    "▪",
    "□",
    "✓",
    "*",
)


OCR_REPLACEMENTS = {
    "1§Е": "IgE",
    "1дЕ": "IgE",
    "ТЬ2": "Th2",
    "8р02": "SpO2",
    "Ра02": "PaO2",
    "РаС02": "PaCO2",
    "РЮ2": "FiO2",
    "РеЫО": "FeNO",
    "01ЫА": "GINA",
}


TRASH_CHARS = (
    "￾",
    "■",
    "",
    "",
    "",
)


def normalize_spaces(text: str) -> str:
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\u00a0", " ")
    text = text.replace("\u00ad", "")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def is_toc_line(text: str) -> bool:
    text = normalize_spaces(text)

    # строка оглавления с точками и номером страницы:
    # "2.1 Жалобы и анамнез................26"
    return bool(re.search(r"\.{3,}\s*\d+\s*$", text))


def is_page_number_line(text: str) -> bool:
    """
    Отсекает отдельные номера страниц:
    7
    8
    13
    31
    """
    text = normalize_spaces(text)
    return bool(re.fullmatch(r"\d{1,3}", text))


def apply_ocr_replacements(text: str) -> str:
    for old, new in OCR_REPLACEMENTS.items():
        text = text.replace(old, new)

    for ch in TRASH_CHARS:
        text = text.replace(ch, "")

    return text


def clean_line_text(text: str) -> str:
    text = apply_ocr_replacements(text)
    return normalize_spaces(text)


def clean_text(text: str) -> str:
    """
    Чистит общий текст, но сохраняет структуру строк.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Склейка переносов слов через дефис:
    # ВИЧ-\nинфекция -> ВИЧ-инфекция
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1-\2", text)

    text = apply_ocr_replacements(text)

    cleaned_lines = []

    for line in text.split("\n"):
        cleaned_lines.append(normalize_spaces(line))

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [138]:
# =========================
# ИЗВЛЕЧЕНИЕ PDF С LAYOUT
# =========================

def is_bold_span(span: Dict[str, Any]) -> bool:
    font = str(span.get("font", "")).lower()
    flags = int(span.get("flags", 0))

    return (
        "bold" in font
        or "black" in font
        or "semibold" in font
        or "demibold" in font
        or bool(flags & 16)
    )


def normalize_bbox(bbox) -> Tuple[float, float, float, float]:
    if not bbox:
        return 0.0, 0.0, 0.0, 0.0

    x0, y0, x1, y1 = bbox
    return float(x0), float(y0), float(x1), float(y1)


def extract_pages_with_layout(pdf_path: Path) -> List[Dict[str, Any]]:
    """
    Извлекает из PDF строки с координатами, размером шрифта и жирностью.
    """
    doc = pymupdf.open(pdf_path)
    pages = []

    for page_index, page in enumerate(doc):
        page_num = page_index + 1
        page_dict = page.get_text("dict", sort=True)

        lines = []

        for block in page_dict.get("blocks", []):
            if block.get("type") != 0:
                continue

            for raw_line in block.get("lines", []):
                spans = raw_line.get("spans", [])

                text_parts = []
                sizes = []
                bold_flags = []

                for span in spans:
                    span_text = span.get("text", "")

                    if not span_text:
                        continue

                    text_parts.append(span_text)
                    sizes.append(float(span.get("size", 0)))
                    bold_flags.append(is_bold_span(span))

                line_text = clean_line_text("".join(text_parts))

                if not line_text:
                    continue

                x0, y0, x1, y1 = normalize_bbox(raw_line.get("bbox"))

                lines.append({
                    "page": page_num,
                    "text": line_text,
                    "bbox": (x0, y0, x1, y1),
                    "max_size": max(sizes) if sizes else 0,
                    "avg_size": sum(sizes) / len(sizes) if sizes else 0,
                    "is_bold": any(bold_flags),
                })

        pages.append({
            "page": page_num,
            "width": float(page.rect.width),
            "height": float(page.rect.height),
            "lines": lines,
            "text": "\n".join(line["text"] for line in lines),
        })

    return pages


def estimate_body_size(pages: List[Dict[str, Any]]) -> float:
    """
    Определяет основной размер шрифта body-текста.
    Берём самый частый размер среди строк.
    """
    sizes = []

    for page in pages:
        for line in page["lines"]:
            size = line.get("max_size", 0)

            if 6 <= size <= 20:
                sizes.append(round(size, 1))

    if not sizes:
        return 10.0

    counter = Counter(sizes)
    return counter.most_common(1)[0][0]

In [139]:
# =========================
# МЕТАДАННЫЕ
# =========================

def extract_cr_id_from_filename(pdf_path: Path) -> Optional[str]:
    """
    КР621_3.pdf -> 621_3
    КР79_2.pdf -> 79_2
    """
    match = re.search(r"КР\s*(\d+(?:_\d+)?)", pdf_path.stem, flags=re.IGNORECASE)

    if match:
        return match.group(1)

    match = re.search(r"(\d+(?:_\d+)?)", pdf_path.stem)

    if match:
        return match.group(1)

    return None


def extract_metadata(text: str) -> Dict[str, Any]:
    metadata = {
        "source_file": None,
        "document_type": "clinical_recommendation",
        "title": None,
        "id": None,
        "year": None,
        "age_group": None,
        "mkb_codes": None,
    }

    lines = [line.strip() for line in text.split("\n") if line.strip()]

    # title
    for i, line in enumerate(lines):
        if "клинические рекомендации" in line.lower():
            for next_line in lines[i + 1:i + 6]:
                if len(next_line) > 5 and not next_line.lower().startswith("год"):
                    metadata["title"] = normalize_spaces(next_line)
                    break
            break

    # year
    year_match = re.search(r"\b(20\d{2})\b", text)
    if year_match:
        metadata["year"] = year_match.group(1)

    # age_group
    age_match = re.search(
        r"Возрастная\s+категория\s*[:\-]?\s*(Взрослые|Дети|Взрослые\s+и\s+дети)",
        text,
        flags=re.IGNORECASE,
    )
    if age_match:
        metadata["age_group"] = normalize_spaces(age_match.group(1))

    # mkb_codes
    mkb_match = re.search(
        r"(?:МКБ-10|МКБ\s*10|код(?:ы)?\s+по\s+МКБ)[^\n:]*[:\-]?\s*([A-ZА-Я]\d{2}(?:\.\d+)?(?:\s*,\s*[A-ZА-Я]?\d{2}(?:\.\d+)?)*)",
        text,
        flags=re.IGNORECASE,
    )
    if mkb_match:
        metadata["mkb_codes"] = normalize_spaces(mkb_match.group(1))

    return metadata


def value_to_str(value: Any) -> Optional[str]:
    if value is None:
        return None

    text = normalize_spaces(str(value))

    if not text:
        return None

    return text


def find_header_row(rows: List[tuple]) -> Optional[int]:
    """
    Ищет строку с заголовками в Excel.
    """
    for idx, row in enumerate(rows[:30]):
        values = [normalize_spaces(str(cell or "")).lower() for cell in row]

        if "id" in values and any("наименование" in v for v in values):
            return idx

    return None


def load_clinical_registry(xlsx_path: Path) -> Dict[str, Dict[str, Any]]:
    """
    Загружает Excel-реестр клинических рекомендаций.
    Ключ словаря — ID клинической рекомендации.
    """
    if not xlsx_path or not xlsx_path.exists():
        return {}

    wb = load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb.active

    rows = list(ws.iter_rows(values_only=True))

    if not rows:
        return {}

    header_idx = find_header_row(rows)

    if header_idx is None:
        return {}

    headers = [normalize_spaces(str(h or "")) for h in rows[header_idx]]

    registry = {}

    for row in rows[header_idx + 1:]:
        item = {}

        for header, value in zip(headers, row):
            if not header:
                continue

            item[header] = value

        raw_id = (
            item.get("ID")
            or item.get("Id")
            or item.get("id")
        )

        if raw_id is None:
            continue

        cr_id = normalize_spaces(str(raw_id))

        if not cr_id:
            continue

        registry[cr_id] = item

    return registry


def get_registry_value(row: Dict[str, Any], candidates: List[str]) -> Optional[str]:
    for candidate in candidates:
        for key, value in row.items():
            if normalize_spaces(key).lower() == candidate.lower():
                return value_to_str(value)

    return None


def find_registry_row(
    registry: Dict[str, Dict[str, Any]],
    cr_id: str
) -> tuple[Optional[str], Optional[Dict[str, Any]]]:
    if cr_id in registry:
        return cr_id, registry[cr_id]

    for registry_id, row in registry.items():
        if registry_id.startswith(cr_id + "_"):
            return registry_id, row

        if cr_id.startswith(registry_id + "_"):
            return registry_id, row

    return None, None


def is_bad_metadata_title(title: Optional[str]) -> bool:
    if not title:
        return True

    title_lower = normalize_spaces(title).lower()

    bad_fragments = (
        "минздрав россии. url",
        "url:",
        "http",
        "www.",
        "клинические рекомендации",
    )

    if len(title_lower) < 5:
        return True

    return any(fragment in title_lower for fragment in bad_fragments)


def extract_year_from_value(value: Any) -> Optional[str]:
    if value is None:
        return None

    match = re.search(r"\b(20\d{2})\b", str(value))

    if match:
        return match.group(1)

    return None


def enrich_metadata_from_registry(
    metadata: Dict[str, Any],
    pdf_path: Path,
    registry: Dict[str, Dict[str, Any]]
) -> Dict[str, Any]:
    """
    Дополняет metadata из Excel-реестра.

    В коде №3 реестр считается более надёжным источником,
    чем PDF, потому что титульная страница PDF может быть сканом.
    """
    cr_id = metadata.get("id") or extract_cr_id_from_filename(pdf_path)

    if not cr_id:
        return metadata

    cr_id = str(cr_id)

    registry_id, row = find_registry_row(registry, cr_id)

    if row is None:
        metadata["id"] = metadata.get("id") or cr_id
        return metadata

    metadata["id"] = registry_id or cr_id

    registry_title = get_registry_value(
        row,
        ["Наименование", "Название", "Клиническая рекомендация"]
    )

    registry_mkb = get_registry_value(
        row,
        ["МКБ-10", "МКБ 10", "Код МКБ", "Коды МКБ"]
    )

    registry_age_group = get_registry_value(
        row,
        ["Возрастная категория", "Возраст"]
    )

    registry_developer = get_registry_value(
        row,
        ["Разработчик"]
    )

    registry_approval_status = get_registry_value(
        row,
        ["Статус одобрения НПС", "Статус одобрения"]
    )

    registry_publication_date = get_registry_value(
        row,
        ["Дата размещения", "Дата публикации", "Дата утверждения"]
    )

    registry_application_status = get_registry_value(
        row,
        ["Статус применения", "Статус"]
    )

    registry_year = (
        get_registry_value(row, ["Год утверждения", "Год"])
        or extract_year_from_value(registry_publication_date)
    )

    # Реестр имеет приоритет
    if registry_title:
        metadata["title"] = registry_title

    if registry_mkb:
        metadata["mkb_codes"] = registry_mkb

    if registry_age_group:
        metadata["age_group"] = registry_age_group

    if registry_year:
        metadata["year"] = registry_year
    elif is_bad_metadata_title(metadata.get("title")):
        metadata["year"] = None

    metadata["developer"] = registry_developer
    metadata["approval_status"] = registry_approval_status
    metadata["publication_date"] = registry_publication_date
    metadata["application_status"] = registry_application_status

    return metadata

In [140]:
# =========================
# ЗАГОЛОВКИ
# =========================

def is_valid_main_heading_title(title: str) -> bool:
    title = normalize_spaces(title).lower()

    if title.startswith(FORBIDDEN_HEADING_STARTS):
        return False

    return title.startswith(MAIN_HEADING_PREFIXES)


def number_to_tuple(number: str) -> Tuple[int, ...]:
    return tuple(int(part) for part in number.split(".") if part.isdigit())


def parse_heading(line: str) -> Optional[Dict[str, Any]]:
    """
    Парсит нумерованный заголовок.
    Поддерживает:
    1.
    1.1
    7.3.3
    7. 3.3.

    Не принимает даты:
    11.11.2020 №60847
    """
    line = normalize_spaces(line)

    if not line:
        return None

    if line.startswith(FORBIDDEN_HEADING_STARTS):
        return None

    match = re.match(
        r"^(\d+(?:\s*\.\s*\d+){0,4})\s*\.?\s+(.+)$",
        line
    )

    if not match:
        return None

    number_raw = match.group(1)
    title = normalize_spaces(match.group(2))

    if not title:
        return None

    if title.startswith(FORBIDDEN_HEADING_STARTS):
        return None

    number = re.sub(r"\s+", "", number_raw).rstrip(".")

    if is_date_like_number(number):
        return None

    parts = number_to_tuple(number)

    if not parts:
        return None

    # Отсекаем даты/нормативные номера вроде 11.11.2020
    if parts[0] < 1 or parts[0] > 7:
        return None

    level = number.count(".") + 1

    if level > 5:
        return None

    if level == 1:
        if len(title) > 500:
            return None

        if not is_valid_main_heading_title(title):
            return None
    else:
        if len(title) > 300:
            return None

    return {
        "number": number,
        "title": title,
        "level": level,
    }


def is_heading_order_valid(
    new_number: str,
    last_number: Optional[str]
) -> bool:
    if not last_number:
        return True

    try:
        return number_to_tuple(new_number) > number_to_tuple(last_number)
    except Exception:
        return True


def is_visual_heading_line(line: Dict[str, Any], body_size: float) -> bool:
    size = line.get("max_size", 0)
    is_bold = line.get("is_bold", False)

    return (
        size >= body_size * 1.35
        or is_bold
    )


def parse_heading_from_layout_line(
    line: Dict[str, Any],
    body_size: float
) -> Optional[Dict[str, Any]]:
    """
    Заголовок определяется по старому regex + визуальным признакам.

    Визуальные признаки сохраняем в heading["visual_heading"],
    но regex остаётся главным структурным фильтром.
    """
    heading = parse_heading(line["text"])

    if not heading:
        return None

    heading["visual_heading"] = is_visual_heading_line(line, body_size)
    heading["page"] = line["page"]
    heading["bbox"] = line["bbox"]

    return heading


def has_unclosed_bracket(text: str) -> bool:
    return text.count("(") > text.count(")")


def join_heading_title_parts(parts: List[str]) -> str:
    title = " ".join(normalize_spaces(part) for part in parts if part.strip())

    # "ВИЧ- инфекция" -> "ВИЧ-инфекция"
    title = re.sub(r"-\s+", "-", title)

    # "ВИЧ инфекция" -> "ВИЧ-инфекция"
    title = re.sub(r"\bВИЧ\s+(инфекц\w*)", r"ВИЧ-\1", title, flags=re.IGNORECASE)
    title = re.sub(r"\bВИЧ\s+(инфицирован\w*)", r"ВИЧ-\1", title, flags=re.IGNORECASE)
    title = re.sub(r"\bВИЧ\s+(инфицир\w*)", r"ВИЧ-\1", title, flags=re.IGNORECASE)

    # Убираем пробелы перед пунктуацией
    title = re.sub(r"\s+([,.;:])", r"\1", title)

    return normalize_spaces(title)


def normalize_heading_key(title: str) -> str:
    """
    Нормализует заголовок для сравнения с оглавлением.
    """
    title = normalize_spaces(title).lower()
    title = title.replace("ё", "е")

    # Убираем точки-лидеры и номер страницы в конце строки TOC
    title = re.sub(r"\.{2,}\s*\d+\s*$", "", title)
    title = re.sub(r"\s+\d+\s*$", "", title)

    title = re.sub(r"[.,;:]+$", "", title)
    title = re.sub(r"\s+", " ", title)

    return title.strip()


def clean_toc_line(line: str) -> str:
    line = normalize_spaces(line)
    line = re.sub(r"\.{2,}\s*\d+\s*$", "", line)
    line = re.sub(r"\s+\d+\s*$", "", line)
    return normalize_spaces(line)


def prettify_toc_title(title: str) -> str:
    """
    Приводит заголовки из оглавления к более нормальному виду.
    Если строка вся в верхнем регистре, делаем sentence case.
    """
    title = normalize_spaces(title)

    if title and title.upper() == title:
        title = title.lower()
        title = title[0].upper() + title[1:]

    return title


def build_toc_maps(front_matter_text: str) -> tuple[Dict[str, str], Dict[str, str]]:
    """
    Возвращает:
    1) title_to_number:
       "консервативное лечение" -> "3.1"

    2) number_to_title:
       "3.1" -> "Консервативное лечение"
    """
    lines = [
        normalize_spaces(line)
        for line in front_matter_text.split("\n")
        if normalize_spaces(line)
    ]

    toc_started = False
    toc_lines = []

    for line in lines:
        line_lower = line.lower()

        if line_lower == "оглавление":
            toc_started = True
            continue

        if not toc_started:
            continue

        # Оглавление обычно заканчивается после приложений.
        if toc_lines and line_lower in {
            "список сокращений",
            "термины и определения",
            "ключевые слова"
        }:
            if any(l.lower().startswith("приложение") for l in toc_lines):
                break

        toc_lines.append(line)

    title_to_number = {}
    number_to_title = {}

    pending_number = None
    pending_title_parts = []

    def flush_pending():
        nonlocal pending_number, pending_title_parts

        if pending_number and pending_title_parts:
            title = clean_toc_line(" ".join(pending_title_parts))
            title = prettify_toc_title(title)

            key = normalize_heading_key(title)

            if key:
                title_to_number[key] = pending_number
                number_to_title[pending_number] = title

        pending_number = None
        pending_title_parts = []

    for line in toc_lines:
        cleaned = clean_toc_line(line)

        match = re.match(
            r"^(\d+(?:\.\d+){0,4})\.?\s+(.+)$",
            cleaned
        )

        if match:
            flush_pending()

            number = match.group(1).rstrip(".")
            title = match.group(2)

            if not is_date_like_number(number):
                pending_number = number
                pending_title_parts = [title]

            continue

        if pending_number:
            if not re.match(
                r"^(Список литературы|Приложение)\b",
                cleaned,
                flags=re.IGNORECASE
            ):
                pending_title_parts.append(cleaned)
            else:
                flush_pending()

    flush_pending()

    return title_to_number, number_to_title

    def flush_pending():
        nonlocal pending_number, pending_title_parts

        if pending_number and pending_title_parts:
            title = clean_toc_line(" ".join(pending_title_parts))
            key = normalize_heading_key(title)

            if key:
                title_map[key] = pending_number

        pending_number = None
        pending_title_parts = []

    for line in toc_lines:
        cleaned = clean_toc_line(line)

        match = re.match(
            r"^(\d+(?:\.\d+){0,4})\.?\s+(.+)$",
            cleaned
        )

        if match:
            flush_pending()

            number = match.group(1).rstrip(".")
            title = match.group(2)

            if not is_date_like_number(number):
                pending_number = number
                pending_title_parts = [title]

            continue

        # Продолжение длинного заголовка в оглавлении
        if pending_number:
            # Если строка не похожа на приложение/список литературы, доклеиваем
            if not re.match(r"^(Список литературы|Приложение)\b", cleaned, flags=re.IGNORECASE):
                pending_title_parts.append(cleaned)
            else:
                flush_pending()

    flush_pending()

    return title_map


def find_toc_number_for_title(
    title: str,
    toc_title_map: Dict[str, str],
    allow_fuzzy: bool = True
) -> Optional[str]:
    raw_title = normalize_spaces(title)
    key = normalize_heading_key(raw_title)

    if not key or len(key) < 5:
        return None

    if key in toc_title_map:
        return toc_title_map[key]

    if not allow_fuzzy:
        return None

    # Строки-продолжения вроде "и противопоказания..."
    # не должны становиться самостоятельными заголовками.
    if raw_title and raw_title[0].islower():
        return None

    if len(key) < 14:
        return None

    best_number = None
    best_score = 0.0

    for toc_key, number in toc_title_map.items():
        score = SequenceMatcher(None, key, toc_key).ratio()

        if score > best_score:
            best_score = score
            best_number = number

    if best_score >= 0.88:
        return best_number

    return None


def apply_toc_number_override(
    heading: Dict[str, Any],
    toc_title_map: Dict[str, str],
    last_heading_number: Optional[str]
) -> Dict[str, Any]:
    """
    Если title найден в оглавлении, используем номер из оглавления.

    Пример:
    body ошибочно дал "3.1 Лазерное лечение",
    TOC знает, что "Лазерное лечение" = 3.2.
    """
    toc_number = find_toc_number_for_title(
        heading.get("title", ""),
        toc_title_map,
        allow_fuzzy=True
    )

    if not toc_number:
        return heading

    if not is_heading_order_valid(toc_number, last_heading_number):
        return heading

    heading["number"] = toc_number
    heading["level"] = toc_number.count(".") + 1

    return heading


def parse_unnumbered_heading_from_toc(
    line: Dict[str, Any],
    toc_title_map: Dict[str, str],
    body_size: float,
    last_heading_number: Optional[str]
) -> Optional[Dict[str, Any]]:
    text = normalize_spaces(line["text"])

    if not text:
        return None

    if parse_heading(text) or parse_heading_number_only(text):
        return None

    if text.startswith(FORBIDDEN_HEADING_STARTS):
        return None

    if len(text) > 120:
        return None

    key = normalize_heading_key(text)

    toc_number = toc_title_map.get(key)

    if not toc_number:
        return None

    if not is_heading_order_valid(toc_number, last_heading_number):
        return None

    return {
        "number": toc_number,
        "title": text,
        "level": toc_number.count(".") + 1,
        "page": line["page"],
        "bbox": line["bbox"],
        "visual_heading": is_visual_heading_line(line, body_size),
    }


def is_heading_continuation(
    line: str,
    heading: Dict[str, Any],
    current_title: str
) -> bool:
    line = normalize_spaces(line)

    if not line:
        return False

    if parse_heading(line):
        return False

    if parse_heading_number_only(line):
        return False

    if re.match(
        r"^(Таблица|Рисунок|Список литературы|Приложение|Ключевые слова|Список сокращений|Оглавление)\b",
        line,
        flags=re.IGNORECASE
    ):
        return False

    if line.startswith(FORBIDDEN_HEADING_STARTS):
        return False

    if len(line) > 320:
        return False

    current_title = normalize_spaces(current_title)
    line_lower = line.lower()

    if current_title.endswith(("-", ",", "(", "/", " к", " и", " или", " по", " при", " с", " в", " для")):
        return True

    if has_unclosed_bracket(current_title):
        return True

    continuation_prefixes = (
        "или ",
        "и ",
        "в том числе",
        "основанных",
        "использовании",
        "применению",
        "к применению",
        "показания",
        "медицинские показания",
        "противопоказания",
        "и противопоказания",
        "методов",
        "метода",
        "методов диагностики",
        "методов лечения",
        "методов профилактики",
        "методов реабилитации",
        "диагностики",
        "лечения",
        "профилактики",
        "реабилитации",
        "инфекция",
        "инфекции",
        "инфицированных",
        "инфицированным",
        "инфицированные",
        "инфицированного",
        "заболевания",
        "состояний",
        "заболеваний",
    )

    if line_lower.startswith(continuation_prefixes):
        return True

    # Для длинных заголовков верхнего уровня разрешаем доклеивать
    # строку, если она выглядит как продолжение, даже если она не начинается со строчной буквы.
    if heading.get("level") == 1 and len(line) < 260:
        if any(word in line_lower for word in ("показания", "противопоказания", "применению", "методов")):
            return True

    if line[0].islower() and len(line) < 220:
        return True

    return False


def should_continue_heading_by_layout(
    line: Dict[str, Any],
    state: Dict[str, Any],
    body_size: float
) -> bool:
    pending = state.get("heading")

    if not pending:
        return False

    if pending.get("extra_lines_used", 0) >= 5:
        return False

    text = normalize_spaces(line["text"])

    if not text:
        return False

    # Новый нормальный заголовок — не продолжение
    if parse_heading(text):
        return False

    # Новый номер заголовка без title — тоже не продолжение
    if parse_heading_number_only(text):
        return False

    current_title = join_heading_title_parts(pending["title_parts"])

    if is_heading_continuation(
        line=text,
        heading=pending["heading"],
        current_title=current_title
    ):
        return True

    # Для длинных заголовков верхнего уровня иногда продолжение тоже жирное/крупное
    if pending["heading"].get("level") == 1:
        text_lower = text.lower()

        if len(text) < 260 and any(
            marker in text_lower
            for marker in (
                "показания",
                "противопоказания",
                "применению",
                "методов",
                "заболеваний",
                "состояний",
            )
        ):
            return True

    return False


def flush_pending_heading(
    state: Dict[str, Any],
    sections: List[Dict[str, Any]],
    last_heading_number: Optional[str],
    toc_number_to_title: Optional[Dict[str, str]] = None
) -> tuple[Optional[Dict[str, Any]], Optional[str]]:
    pending = state.get("heading")

    if not pending:
        return None, last_heading_number

    heading = pending["heading"]
    collected_title = join_heading_title_parts(pending["title_parts"])

    toc_title = None

    if toc_number_to_title:
        toc_title = toc_number_to_title.get(heading["number"])

    # Если title из body короче, чем title из оглавления,
    # берём полный title из оглавления.
    if toc_title:
        collected_key = normalize_heading_key(collected_title)
        toc_key = normalize_heading_key(toc_title)

        if collected_key and toc_key:
            if collected_key in toc_key and len(toc_key) > len(collected_key):
                heading["title"] = toc_title
            else:
                heading["title"] = collected_title
        else:
            heading["title"] = collected_title
    else:
        heading["title"] = collected_title

    section = {
        "number": heading["number"],
        "title": heading["title"],
        "level": heading["level"],
        "text": "",
        "children": [],
    }

    sections.append(section)

    last_heading_number = heading["number"]
    state["heading"] = None

    return section, last_heading_number


def is_date_like_number(number: str) -> bool:
    """
    Отсекает даты вида:
    11.11.2020
    28.01.2021
    """
    return bool(re.fullmatch(r"\d{1,2}\.\d{1,2}\.\d{4}", number))


def parse_heading_number_only(line: str) -> Optional[Dict[str, Any]]:
    """
    Ловит случаи, когда PyMuPDF разделил заголовок так:

    1.1
    Определение заболевания или состояния ...

    Вместо одной строки:
    1.1 Определение заболевания или состояния ...
    """
    line = normalize_spaces(line)

    if not line:
        return None

    if line.startswith(FORBIDDEN_HEADING_STARTS):
        return None

    match = re.fullmatch(
        r"(\d+(?:\s*\.\s*\d+){0,4})\.?",
        line
    )

    if not match:
        return None

    number = re.sub(r"\s+", "", match.group(1)).rstrip(".")

    if is_date_like_number(number):
        return None

    parts = number_to_tuple(number)

    if not parts:
        return None

    # Для клинических рекомендаций основные разделы обычно 1–7.
    # Это также отсекает даты и номера приказов.
    if parts[0] < 1 or parts[0] > 7:
        return None

    level = number.count(".") + 1

    if level > 5:
        return None

    return {
        "number": number,
        "title": "",
        "level": level,
    }


def should_attach_title_to_number_only(
    line: Dict[str, Any],
    pending_number: Dict[str, Any],
    body_size: float
) -> bool:
    """
    Проверяет, можно ли следующую строку считать title
    для предыдущего номера заголовка.
    """
    text = normalize_spaces(line["text"])

    if not text:
        return False

    if parse_heading(text):
        return False

    if parse_heading_number_only(text):
        return False

    if text.startswith(FORBIDDEN_HEADING_STARTS):
        return False

    if re.match(
        r"^(Таблица|Рисунок|Список литературы|Приложение|Оглавление)\b",
        text,
        flags=re.IGNORECASE
    ):
        return False

    level = pending_number["level"]

    if level == 1:
        return is_valid_main_heading_title(text)

    if len(text) > 300:
        return False

    # Для подзаголовков title обычно жирный/крупный
    # или начинается с заглавной буквы.
    return (
        is_visual_heading_line(line, body_size)
        or text[0].isupper()
    )


def clone_line_with_text(line: Dict[str, Any], text: str) -> Dict[str, Any]:
    new_line = dict(line)
    new_line["text"] = normalize_spaces(text)
    return new_line


def split_line_by_inline_numbered_heading(line: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Делит строку, если внутри неё встретился новый нумерованный заголовок.

    Пример:
    "... [75]. 4. Медицинская реабилитация ..."

    станет:
    "... [75]."
    "4. Медицинская реабилитация ..."
    """
    text = normalize_spaces(line["text"])

    if not text:
        return []

    pattern = re.compile(
        r"\s+(?=(?:[1-7](?:\.\d+){0,4})\.?\s+"
        r"(?:Краткая|Диагностика|Лечение|Медицинская|Профилактика|Организация|Дополнительная|Критерии|"
        r"Жалобы|Физикальное|Лабораторные|Инструментальные|Иные|Консервативное|Лазерное|Хирургическое|Диетотерапия))"
    )

    parts = pattern.split(text)

    if len(parts) <= 1:
        return [line]

    return [
        clone_line_with_text(line, part)
        for part in parts
        if normalize_spaces(part)
    ]


def split_line_by_inline_toc_heading(
    line: Dict[str, Any],
    toc_title_map: Dict[str, str]
) -> List[Dict[str, Any]]:
    """
    Делит строку, если внутри неё встретился ненумерованный заголовок из TOC.

    Пример:
    "... [69]. Консервативное лечение Рекомендуется ..."

    станет:
    "... [69]."
    "Консервативное лечение"
    "Рекомендуется ..."
    """
    text = normalize_spaces(line["text"])

    if not text:
        return []

    # Берём короткие заголовки из оглавления.
    # Например: "Консервативное лечение", "Лазерное лечение", "Диетотерапия".
    toc_titles = sorted(
        [
            key for key in toc_title_map.keys()
            if 5 <= len(key) <= 80
        ],
        key=len,
        reverse=True
    )

    current_parts = [text]

    for toc_key in toc_titles:
        # Не режем слишком общие/длинные заголовки
        if len(toc_key.split()) > 5:
            continue

        new_parts = []

        for part in current_parts:
            part_norm = normalize_heading_key(part)

            # Если весь part и есть заголовок, не режем
            if part_norm == toc_key:
                new_parts.append(part)
                continue

            # Ищем заголовок как отдельный фрагмент после конца предложения.
            pattern = re.compile(
                r"(.+?[\].])\s+(" + re.escape(toc_key) + r")\s+(.+)",
                flags=re.IGNORECASE
            )

            match = pattern.match(part.lower())

            if not match:
                new_parts.append(part)
                continue

            before = part[:len(match.group(1))]
            heading = part[len(match.group(1)):len(match.group(1)) + 1 + len(match.group(2))].strip()
            after = part[len(match.group(1)) + 1 + len(match.group(2)):].strip()

            new_parts.extend([before, heading, after])

        current_parts = [
            normalize_spaces(p)
            for p in new_parts
            if normalize_spaces(p)
        ]

    return [
        clone_line_with_text(line, part)
        for part in current_parts
        if normalize_spaces(part)
    ]


def split_inline_headings_in_lines(
    lines: List[Dict[str, Any]],
    toc_title_map: Dict[str, str]
) -> List[Dict[str, Any]]:
    result = []

    for line in lines:
        first_split = split_line_by_inline_numbered_heading(line)

        for part_line in first_split:
            second_split = split_line_by_inline_toc_heading(
                part_line,
                toc_title_map
            )
            result.extend(second_split)

    return result

In [141]:
# =========================
# ПОИСК РЕАЛЬНОГО НАЧАЛА РАЗДЕЛОВ
# =========================

def is_probably_body_line(line: str) -> bool:
    line = normalize_spaces(line)

    if not line:
        return False

    if parse_heading(line):
        return False

    if re.fullmatch(r"\d+", line):
        return False

    if len(line) < 40:
        return False

    return bool(re.search(r"[А-Яа-яA-Za-z]", line))


def find_real_sections_start(lines: List[str]) -> int:
    """
    Ищет реальное начало основного текста, а не пункт оглавления.
    Обычно это первый настоящий раздел 1, после которого идёт 1.1 и body-текст.
    """
    for i, line in enumerate(lines):
        heading = parse_heading(line)

        if not heading:
            continue

        if heading["number"] != "1":
            continue

        found_1_1_index = None

        for j in range(i + 1, min(i + 25, len(lines))):
            next_heading = parse_heading(lines[j])

            if next_heading and next_heading["number"] == "1.1":
                found_1_1_index = j
                break

        if found_1_1_index is None:
            continue

        for k in range(found_1_1_index + 1, min(found_1_1_index + 10, len(lines))):
            next_heading = parse_heading(lines[k])

            if next_heading and next_heading["number"].startswith("1.2"):
                break

            if is_probably_body_line(lines[k]):
                return i

    for i, line in enumerate(lines):
        heading = parse_heading(line)

        if heading and heading["number"] == "1":
            return i

    return 0

In [142]:
# =========================
# EXCLUDED-БЛОКИ
# =========================

def detect_excluded_type(line: str) -> Optional[str]:
    line_lower = normalize_spaces(line).lower()

    if line_lower == "оглавление":
        return "toc"

    if line_lower.startswith("список литературы"):
        return "references"

    if line_lower.startswith("приложение"):
        return "appendices"

    return None


def split_front_matter(front_matter_text: str) -> Dict[str, List[Dict[str, str]]]:
    """
    Делит предтекстовую часть на front_matter и toc.

    Важно:
    первые "Ключевые слова", "Список сокращений", "Термины и определения"
    после слова "Оглавление" считаются пунктами оглавления.
    """
    result = {
        "front_matter": [],
        "toc": [],
    }

    lines = front_matter_text.split("\n")

    front_lines = []
    toc_lines = []

    current_block = "front_matter"
    toc_title = "Оглавление"

    seen_toc_tail = False

    for line in lines:
        line = normalize_spaces(line)

        if not line:
            continue

        line_lower = line.lower()

        if line_lower == "оглавление":
            current_block = "toc"
            toc_title = line
            continue

        if current_block == "toc":
            if line_lower.startswith("список литературы") or line_lower.startswith("приложение"):
                seen_toc_tail = True

            if seen_toc_tail and line_lower in {
                "ключевые слова",
                "список сокращений",
                "термины и определения",
            }:
                current_block = "front_matter"
                front_lines.append(line)
                continue

            toc_lines.append(line)
            continue

        front_lines.append(line)

    if front_lines:
        result["front_matter"].append({
            "title": "front_matter",
            "text": normalize_spaces(" ".join(front_lines)),
        })

    if toc_lines:
        result["toc"].append({
            "title": toc_title,
            "text": normalize_spaces(" ".join(toc_lines)),
        })

    return result

In [143]:
# =========================
# ТАБЛИЦЫ ПО BBOX
# =========================

def bbox_inside_page(
    bbox: Tuple[float, float, float, float],
    page_width: float,
    page_height: float
) -> bool:
    x0, y0, x1, y1 = bbox

    return (
        0 <= x0 < x1 <= page_width
        and 0 <= y0 < y1 <= page_height
    )


def count_non_empty_cells(rows: List[List[Any]]) -> int:
    count = 0

    for row in rows:
        for cell in row:
            if cell is not None and normalize_spaces(str(cell)):
                count += 1

    return count


def is_valid_table(
    bbox: Tuple[float, float, float, float],
    rows: List[List[Any]],
    page_width: float,
    page_height: float
) -> bool:
    if not bbox_inside_page(bbox, page_width, page_height):
        return False

    row_count = len(rows)
    col_count = max((len(row) for row in rows), default=0)

    if row_count < 2 or col_count < 2:
        return False

    if count_non_empty_cells(rows) < 4:
        return False

    return True


def parse_table_caption(text: str) -> tuple[Optional[str], Optional[str]]:
    match = re.match(
        r"^\s*Таблица\s+(\d+(?:\.\d+)?)\.?\s*(.*)$",
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return None, None

    number = match.group(1)
    caption = normalize_spaces(text)

    return number, caption


def attach_table_captions(
    tables_by_page: Dict[int, List[Dict[str, Any]]],
    pages: List[Dict[str, Any]]
) -> None:
    """
    Привязывает подписи вида "Таблица N ..." к найденным bbox-таблицам.
    """
    pages_by_number = {page["page"]: page for page in pages}

    for page_num, tables in tables_by_page.items():
        page = pages_by_number.get(page_num)

        if not page:
            continue

        caption_lines = []

        for line in page["lines"]:
            number, caption = parse_table_caption(line["text"])

            if number:
                caption_lines.append({
                    "number": number,
                    "caption": caption,
                    "bbox": line["bbox"],
                })

        for table in tables:
            tx0, ty0, tx1, ty1 = table["bbox"]

            candidates = []

            for caption_line in caption_lines:
                cx0, cy0, cx1, cy1 = caption_line["bbox"]

                # Подпись обычно находится чуть выше таблицы
                if cy1 <= ty0 + 10 and ty0 - cy1 <= 120:
                    candidates.append(caption_line)

            if candidates:
                best = min(
                    candidates,
                    key=lambda c: abs(ty0 - c["bbox"][3])
                )

                table["number"] = best["number"]
                table["caption"] = best["caption"]


def extract_tables_with_bboxes(
    pdf_path: Path,
    pages: List[Dict[str, Any]]
) -> Dict[int, List[Dict[str, Any]]]:
    doc = pymupdf.open(pdf_path)
    tables_by_page = {}

    for page_index, page in enumerate(doc):
        page_num = page_index + 1
        page_tables = []

        try:
            finder = page.find_tables()
            found_tables = getattr(finder, "tables", [])
        except Exception:
            found_tables = []

        for table_index, table in enumerate(found_tables, start=1):
            bbox = normalize_bbox(table.bbox)

            try:
                rows = table.extract()
            except Exception:
                rows = []

            if not is_valid_table(
                bbox=bbox,
                rows=rows,
                page_width=float(page.rect.width),
                page_height=float(page.rect.height),
            ):
                continue

            page_tables.append({
                "page": page_num,
                "number": None,
                "caption": None,
                "bbox": bbox,
                "shape": {
                    "rows": len(rows),
                    "cols": max((len(row) for row in rows), default=0),
                },
                "raw_text": rows,
            })

        tables_by_page[page_num] = page_tables

    attach_table_captions(tables_by_page, pages)

    return tables_by_page


def covered_by_table(
    line_bbox: Tuple[float, float, float, float],
    table_bboxes: List[Tuple[float, float, float, float]]
) -> bool:
    x0, y0, x1, y1 = line_bbox

    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2

    for tx0, ty0, tx1, ty1 in table_bboxes:
        if tx0 <= cx <= tx1 and ty0 <= cy <= ty1:
            return True

    return False


def flatten_tables(tables_by_page: Dict[int, List[Dict[str, Any]]]) -> List[Dict[str, Any]]:
    tables = []

    for page_tables in tables_by_page.values():
        tables.extend(page_tables)

    return tables

In [144]:
# =========================
# ИЕРАРХИЯ SECTIONS
# =========================

def build_section_hierarchy(flat_sections: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    root_sections = []
    stack = []

    for section in flat_sections:
        section["children"] = []

        current_level = section["level"]

        while stack and stack[-1]["level"] >= current_level:
            stack.pop()

        if stack:
            stack[-1]["children"].append(section)
        else:
            root_sections.append(section)

        stack.append(section)

    return root_sections


def iter_sections(sections: List[Dict[str, Any]]):
    for section in sections:
        yield section

        for child in section.get("children", []):
            yield from iter_sections([child])

In [145]:
# =========================
# СБОРКА SECTIONS ИЗ LAYOUT
# =========================

def build_sections_from_layout(
    pages: List[Dict[str, Any]],
    tables_by_page: Dict[int, List[Dict[str, Any]]],
    body_size: float
) -> Dict[str, Any]:

    excluded = {
        "front_matter": [],
        "toc": [],
        "references": [],
        "appendices": [],
        "other": [],
    }

    all_lines = []

    for page in pages:
        page_num = page["page"]

        table_bboxes = [
            table["bbox"]
            for table in tables_by_page.get(page_num, [])
        ]

        for line in page["lines"]:
            text = normalize_spaces(line["text"])
            
            if is_page_number_line(text):
                continue

            if covered_by_table(line["bbox"], table_bboxes):
                continue

            all_lines.append(line)

    text_lines = [line["text"] for line in all_lines]

    start_index = find_real_sections_start(text_lines)

    front_matter_text = "\n".join(text_lines[:start_index]).strip()

    toc_title_map, toc_number_to_title = build_toc_maps(front_matter_text)

    # Разрезаем строки основного текста, если внутри строки спрятан заголовок.
    # Например:
    # "... [75]. 4. Медицинская реабилитация ..."
    main_lines = split_inline_headings_in_lines(
        all_lines[start_index:],
        toc_title_map
    )

    all_lines = all_lines[:start_index] + main_lines

    if front_matter_text:
        split_front = split_front_matter(front_matter_text)

        excluded["front_matter"].extend(split_front["front_matter"])
        excluded["toc"].extend(split_front["toc"])

    sections = []

    current_section = None
    current_excluded_type = None
    last_heading_number = None

    state = {
        "heading": None,
        "heading_number_only": None,
    }

    i = start_index

    while i < len(all_lines):
        line = all_lines[i]
        text = normalize_spaces(line["text"])

        if not text:
            i += 1
            continue

        detected_excluded_type = detect_excluded_type(text)

        # Если уже внутри references/appendices,
        # строки больше не проверяются как sections.
        if current_excluded_type:
            if detected_excluded_type in {"references", "appendices"}:
                current_excluded_type = detected_excluded_type

                excluded[current_excluded_type].append({
                    "title": text,
                    "text": "",
                })
            else:
                if excluded[current_excluded_type]:
                    excluded[current_excluded_type][-1]["text"] += " " + text
                else:
                    excluded[current_excluded_type].append({
                        "title": current_excluded_type,
                        "text": text,
                    })

            i += 1
            continue

        # 1. Если до этого была строка только с номером:
        # 4.
        # Медицинская реабилитация...
        pending_number = state.get("heading_number_only")

        if pending_number:
            if should_attach_title_to_number_only(
                line=line,
                pending_number=pending_number,
                body_size=body_size
            ):
                heading = {
                    "number": pending_number["number"],
                    "title": text,
                    "level": pending_number["level"],
                    "page": line["page"],
                    "bbox": line["bbox"],
                    "visual_heading": is_visual_heading_line(line, body_size),
                }

                heading = apply_toc_number_override(
                    heading=heading,
                    toc_title_map=toc_title_map,
                    last_heading_number=last_heading_number
                )

                state["heading"] = {
                    "heading": heading,
                    "title_parts": [heading["title"]],
                    "extra_lines_used": 0,
                }

                state["heading_number_only"] = None
                i += 1
                continue
            else:
                state["heading_number_only"] = None

        # 2. ВАЖНО: сначала проверяем продолжение уже начатого заголовка.
        # Это исправляет:
        # 1. Краткая информация ... (группе
        # заболеваний или состояний)
        #
        # и:
        # 3. Лечение ..., медицинские
        # показания и противопоказания ...
        if should_continue_heading_by_layout(line, state, body_size):
            state["heading"]["title_parts"].append(text)
            state["heading"]["extra_lines_used"] += 1
            i += 1
            continue

        # 3. Начало excluded-блока
        if detected_excluded_type in {"references", "appendices"}:
            current_section, last_heading_number = flush_pending_heading(
                state=state,
                sections=sections,
                last_heading_number=last_heading_number,
                toc_number_to_title=toc_number_to_title,
            )
            
            state["heading_number_only"] = None
            current_section = None
            current_excluded_type = detected_excluded_type

            excluded[current_excluded_type].append({
                "title": text,
                "text": "",
            })

            i += 1
            continue

        # 4. Полноценный заголовок в одной строке:
        # 2.1 Жалобы и анамнез
        heading = parse_heading_from_layout_line(line, body_size)

        if heading:
            heading = apply_toc_number_override(
                heading=heading,
                toc_title_map=toc_title_map,
                last_heading_number=last_heading_number
            )

        if heading and is_heading_order_valid(heading["number"], last_heading_number):
            current_section, last_heading_number = flush_pending_heading(
                state=state,
                sections=sections,
                last_heading_number=last_heading_number,
                toc_number_to_title=toc_number_to_title,
            )

            state["heading_number_only"] = None

            state["heading"] = {
                "heading": heading,
                "title_parts": [heading["title"]],
                "extra_lines_used": 0,
            }

            i += 1
            continue

        # 5. Заголовок только номером:
        # 4.
        # Медицинская реабилитация...
        number_only = parse_heading_number_only(text)

        if number_only and is_heading_order_valid(number_only["number"], last_heading_number):
            # Для верхнего уровня не требуем visual_heading,
            # иначе можно потерять 4, 5, 6, 7.
            if number_only["level"] == 1 or is_visual_heading_line(line, body_size):
                flushed_section, last_heading_number = flush_pending_heading(
                    state=state,
                    sections=sections,
                    last_heading_number=last_heading_number,
                    toc_number_to_title=toc_number_to_title,
                )

                if flushed_section is not None:
                    current_section = flushed_section

                state["heading_number_only"] = number_only

                i += 1
                continue

        # 6. Ненумерованный заголовок, найденный по TOC:
        # Консервативное лечение
        toc_heading = parse_unnumbered_heading_from_toc(
            line=line,
            toc_title_map=toc_title_map,
            body_size=body_size,
            last_heading_number=last_heading_number
        )

        if toc_heading:
            current_section, last_heading_number = flush_pending_heading(
                state=state,
                sections=sections,
                last_heading_number=last_heading_number,
                toc_number_to_title=toc_number_to_title,
            )

            state["heading_number_only"] = None

            state["heading"] = {
                "heading": toc_heading,
                "title_parts": [toc_heading["title"]],
                "extra_lines_used": 0,
            }

            i += 1
            continue

        # 7. Если был накопленный heading, а текущая строка уже обычный текст,
        # закрываем heading и начинаем писать body-текст.
        if state.get("heading"):
            current_section, last_heading_number = flush_pending_heading(
                state=state,
                sections=sections,
                last_heading_number=last_heading_number,
                toc_number_to_title=toc_number_to_title,
            )

        if current_section:
            current_section["text"] += " " + text
        else:
            excluded["other"].append({
                "title": "unassigned",
                "text": text,
            })

        i += 1

    current_section, last_heading_number = flush_pending_heading(
                state=state,
                sections=sections,
                last_heading_number=last_heading_number,
                toc_number_to_title=toc_number_to_title,
            )

    for section in sections:
        section["text"] = normalize_spaces(section.get("text", ""))

    for group in excluded.values():
        for item in group:
            item["text"] = normalize_spaces(item.get("text", ""))

    section_tree = build_section_hierarchy(sections)

    return {
        "sections": section_tree,
        "excluded": excluded,
    }

In [146]:
# =========================
# СТАТИСТИКА
# =========================

def table_to_text(table: Dict[str, Any]) -> str:
    parts = []

    if table.get("caption"):
        parts.append(str(table["caption"]))

    rows = table.get("raw_text") or []

    for row in rows:
        for cell in row:
            if cell is not None:
                cell_text = normalize_spaces(str(cell))

                if cell_text:
                    parts.append(cell_text)

    return " ".join(parts)


def calculate_stats(
    full_text: str,
    sections: List[Dict[str, Any]],
    excluded: Dict[str, Any],
    tables: List[Dict[str, Any]]
) -> Dict[str, Any]:

    total_chars = len(full_text)
    total_words = len(full_text.split())

    all_sections = list(iter_sections(sections))

    included_chars = sum(
        len(section.get("title", "")) + len(section.get("text", ""))
        for section in all_sections
    )

    excluded_chars = 0

    for group in excluded.values():
        for item in group:
            excluded_chars += len(item.get("title", "")) + len(item.get("text", ""))

    table_chars = sum(
        len(table_to_text(table))
        for table in tables
    )

    accounted_chars_raw = included_chars + excluded_chars + table_chars

    # Не даём coverage быть больше 100% из-за дублей таблиц/подписей.
    accounted_chars = min(accounted_chars_raw, total_chars)

    coverage_percent = round(
        accounted_chars / total_chars * 100,
        2
    ) if total_chars else 0

    return {
        "total_chars": total_chars,
        "total_words": total_words,
        "sections_found": len(all_sections),
        "tables_found": len(tables),
        "included_chars": included_chars,
        "excluded_chars": excluded_chars,
        "table_chars": table_chars,
        "accounted_chars": accounted_chars,
        "accounted_chars_raw": accounted_chars_raw,
        "coverage_percent": coverage_percent,
    }

In [147]:
# =========================
# ОСНОВНАЯ ФУНКЦИЯ
# =========================

def parse_clinical_recommendation(
    pdf_path: Path,
    registry_xlsx_path: Optional[Path] = None
) -> Dict[str, Any]:

    pages = extract_pages_with_layout(pdf_path)

    full_text = "\n".join(page["text"] for page in pages)
    clean_full_text = clean_text(full_text)

    metadata = extract_metadata(clean_full_text)
    metadata["source_file"] = pdf_path.name

    if registry_xlsx_path and registry_xlsx_path.exists():
        registry = load_clinical_registry(registry_xlsx_path)

        metadata = enrich_metadata_from_registry(
            metadata=metadata,
            pdf_path=pdf_path,
            registry=registry,
        )
    else:
        metadata["id"] = metadata.get("id") or extract_cr_id_from_filename(pdf_path)

    body_size = estimate_body_size(pages)

    tables_by_page = extract_tables_with_bboxes(
        pdf_path=pdf_path,
        pages=pages,
    )

    section_result = build_sections_from_layout(
        pages=pages,
        tables_by_page=tables_by_page,
        body_size=body_size,
    )

    sections = section_result["sections"]
    excluded = section_result["excluded"]

    tables = flatten_tables(tables_by_page)

    stats = calculate_stats(
        full_text=clean_full_text,
        sections=sections,
        excluded=excluded,
        tables=tables,
    )

    return {
        "metadata": metadata,
        "sections": sections,
        "tables": tables,
        "excluded": excluded,
        "stats": stats,
    }


In [148]:
# =========================
# ЗАПУСК
# =========================

if __name__ == "__main__":
    registry_path = REGISTRY_XLSX_PATH if REGISTRY_XLSX_PATH.exists() else None

    result = parse_clinical_recommendation(
        pdf_path=PDF_PATH,
        registry_xlsx_path=registry_path,
    )

    with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"JSON saved to: {OUTPUT_JSON_PATH}")

JSON saved to: C:\Users\Aleksey\Desktop\Рабочие задачи\Задача к 19.06.2026\тест_структурного_парсера_v2\КР359_3.json
